# Capítulo 5 — Quantização: Reduzindo o Tamanho do Modelo sem Perder Qualidade

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap05_quantizacao.ipynb)

**Fonte original:** [orca3/llm-model-serving — ch06/quantization_3way_300.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch06).

Adaptado e comentado por **Anderson Ejepsen**.

Créditos aos autores originais do repositório [orca3/llm-model-serving](https://github.com/orca3/llm-model-serving).

## Configuração do Ambiente

Este notebook compara três estratégias de quantização para o modelo Qwen2.5-7B-Instruct:

1. **FP16** — Precisão original (16 bits por parâmetro)
2. **GPTQ-Int4** — Quantização pós-treinamento para 4 bits
3. **FP8** — Precisão de 8 bits com escala dinâmica

Usamos o benchmark de serving do vLLM com o dataset ShareGPT para medir TTFT, TPOT e throughput em diferentes níveis de concorrência.

> **Requisitos:** GPU com pelo menos 24GB de VRAM (L4 ou superior). No Colab, selecione GPU L4.

In [ ]:
!pip install vllm==0.8.5.post1

### Verificar GPU disponível e versão do vLLM

In [ ]:
!pip freeze | grep vllm
!nvidia-smi

### Download do dataset e script de benchmark

O dataset ShareGPT contém conversas reais de usuários, simulando cargas de trabalho realistas. O script de benchmark do vLLM mede métricas-chave de desempenho.

In [ ]:
!wget https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json
!git clone https://github.com/vllm-project/vllm.git

### Configuração das variáveis de ambiente

In [ ]:
import torch; print(torch.__version__); print(torch.version.cuda)
import time
import vllm
import os

benchmark_script = "vllm/benchmarks/benchmark_serving.py"

### Função para verificar saúde do servidor vLLM

Após iniciar o servidor vLLM em background, esta função verifica se ele está pronto para receber requisições.

In [ ]:
import time
import requests

def check_vllm_status():
    """Verifica se o servidor vLLM está pronto para receber requisições."""
    max_retries = 100
    wait_time = 10

    print("Aguardando servidor vLLM ficar saudável...")

    for i in range(max_retries):
        try:
            r = requests.get("http://127.0.0.1:8000/health")
            if r.ok:
                print("Servidor vLLM está pronto!")
                break
        except Exception as e:
            print(f"Tentativa {i+1}: Ainda não está pronto...")

        time.sleep(wait_time)
    else:
        print("Timeout ao aguardar servidor vLLM.")
    return

check_vllm_status()

---
## 5.1 — Benchmark: Modelo FP16 (Precisão Original)

Primeiro, servimos o modelo original em FP16 (16 bits). Este é o baseline de qualidade e desempenho.

In [ ]:
hf_model_id = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
# Inicia o servidor vLLM em background com o modelo FP16
!nohup vllm serve {hf_model_id} \
       --disable-log-requests \
       > vllm.log 2>&1 &

In [ ]:
check_vllm_status()

### Benchmark com diferentes níveis de concorrência

Executamos o benchmark com concorrência 1 (sequencial), 10 e 100 para medir como o modelo FP16 se comporta sob diferentes cargas.

In [ ]:
# Concorrência 1 (sequencial)
!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 10 \
  --max-concurrency 1 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

In [ ]:
# Concorrência 10 (moderada)
!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 10 \
  --max-concurrency 10 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

In [ ]:
# Concorrência 100 (alta carga)
!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 100 \
  --max-concurrency 100 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

In [ ]:
# Concorrência 300 (stress test)
!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 300 \
  --max-concurrency 300 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

In [ ]:
# Para o servidor FP16
!pkill -f "vllm serve" && sleep 10

---
## 5.2 — Benchmark: Modelo GPTQ-Int4

O GPTQ (Generalized Post-Training Quantization) reduz os pesos do modelo de 16 bits para 4 bits. Isso reduz o uso de memória em ~4x, mas pode haver uma pequena perda de qualidade. A vantagem é que modelos maiores cabem na mesma GPU.

In [ ]:
hf_model_id = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"  # Modelo quantizado em 4 bits

In [ ]:
!nohup vllm serve {hf_model_id} \
       --disable-log-requests \
       > vllm.log 2>&1 &

In [ ]:
check_vllm_status()

In [ ]:
# Executa os mesmos benchmarks com o modelo GPTQ-Int4
!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 10 \
  --max-concurrency 1 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 10 \
  --max-concurrency 10 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 100 \
  --max-concurrency 100 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 300 \
  --max-concurrency 300 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

In [ ]:
!pkill -f "vllm serve" && sleep 10

---
## 5.3 — Benchmark: Modelo FP8 (Precisão Dinâmica de 8 bits)

O FP8 é um formato de ponto flutuante de 8 bits que mantém melhor qualidade que Int4, com redução de memória de ~2x em relação ao FP16. Requer hardware com suporte nativo (H100, L4).

In [ ]:
hf_model_id = "RedHatAI/Qwen2.5-7B-Instruct-FP8-dynamic"  # Modelo FP8 com escala dinâmica

In [ ]:
!nohup vllm serve {hf_model_id} \
       --disable-log-requests \
       > vllm.log 2>&1 &

In [ ]:
check_vllm_status()

In [ ]:
# Executa os mesmos benchmarks com o modelo FP8
!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 10 \
  --max-concurrency 1 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 10 \
  --max-concurrency 10 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 100 \
  --max-concurrency 100 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

!python3 {benchmark_script} \
  --backend vllm \
  --model {hf_model_id} \
  --endpoint /v1/completions \
  --dataset-name sharegpt \
  --dataset-path /content/ShareGPT_V3_unfiltered_cleaned_split.json \
  --num-prompts 300 \
  --max-concurrency 300 \
  --save-result \
  --append-result \
  --result-filename vllm_bench.json

---
## 5.4 — Visualização Comparativa dos Resultados

Os gráficos abaixo comparam as três estratégias de quantização em três métricas-chave:
- **TTFT (Time to First Token)**: latência até o primeiro token — impacta a experiência do usuário
- **TPOT (Time per Output Token)**: latência por token após o primeiro — define a velocidade de streaming
- **Request Throughput**: requisições por segundo — capacidade do servidor

In [ ]:
import matplotlib.pyplot as plt
import json
import numpy as np

# Carrega os resultados do benchmark
with open("vllm_bench.json", "r") as f:
    data = [json.loads(line) for line in f if line.strip()]

### Gráfico 1: TTFT Mediano por Concorrência e Modelo

O TTFT mede o tempo até o primeiro token ser gerado. Modelos quantizados tendem a ter TTFT menor porque processam a fase de prefill mais rápido (menos dados para mover da memória).

In [ ]:
# Organiza os dados por concorrência
grouped = {}
for entry in data:
    rate = entry["max_concurrency"]
    model = entry["model_id"]
    ttft = entry["median_ttft_ms"]
    if rate not in grouped:
        grouped[rate] = {}
    grouped[rate][model] = ttft

sorted_rates = sorted(grouped.keys())
all_models = sorted(set(model for group in grouped.values() for model in group))

bar_width = 0.2
x = np.arange(len(sorted_rates))
offsets = np.linspace(-bar_width*(len(all_models)-1)/2, bar_width*(len(all_models)-1)/2, len(all_models))

plt.figure(figsize=(10, 6))

for i, model in enumerate(all_models):
    ttft_values = [grouped[rate].get(model, 0) for rate in sorted_rates]
    bar_positions = x + offsets[i]
    bars = plt.bar(bar_positions, ttft_values, bar_width, label=model.split("/")[-1])
    for pos, val in zip(bar_positions, ttft_values):
        if val > 0:
            plt.text(pos, val + 10, f"{val:.1f}", ha='center', va='bottom', fontsize=8)

plt.xlabel("Concorrência Máxima")
plt.ylabel("TTFT Mediano (ms)")
plt.title("TTFT Mediano por Concorrência e Modelo")
plt.xticks(x, sorted_rates)
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.legend(title="Modelo")
plt.tight_layout()
plt.show()

### Gráfico 2: TPOT Mediano por Concorrência e Modelo

O TPOT mede o tempo por token de saída (excluindo o primeiro). Quantização reduz o TPOT porque menos dados precisam ser lidos da memória a cada iteração.

In [ ]:
# Organiza dados para TPOT
grouped = {}
for entry in data:
    rate = entry["max_concurrency"]
    model = entry["model_id"]
    ttft = entry["median_tpot_ms"]
    if rate not in grouped:
        grouped[rate] = {}
    grouped[rate][model] = ttft

sorted_rates = sorted(grouped.keys())
all_models = sorted(set(model for group in grouped.values() for model in group))

bar_width = 0.2
x = np.arange(len(sorted_rates))
offsets = np.linspace(-bar_width*(len(all_models)-1)/2, bar_width*(len(all_models)-1)/2, len(all_models))

plt.figure(figsize=(10, 6))

for i, model in enumerate(all_models):
    ttft_values = [grouped[rate].get(model, 0) for rate in sorted_rates]
    bar_positions = x + offsets[i]
    bars = plt.bar(bar_positions, ttft_values, bar_width, label=model.split("/")[-1])
    for pos, val in zip(bar_positions, ttft_values):
        if val > 0:
            plt.text(pos, val + 1, f"{val:.1f}", ha='center', va='bottom', fontsize=8)

plt.xlabel("Concorrência Máxima")
plt.ylabel("TPOT Mediano (ms)")
plt.title("TPOT Mediano por Concorrência e Modelo")
plt.xticks(x, sorted_rates)
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.legend(title="Modelo")
plt.tight_layout()
plt.show()

### Gráfico 3: Request Throughput por Concorrência e Modelo

O throughput de requisições mostra quantas requisições o servidor pode processar por segundo. Modelos quantizados geralmente oferecem throughput maior porque processam cada requisição mais rápido.

In [ ]:
# Organiza dados para Request Throughput
grouped = {}
for entry in data:
    rate = entry["max_concurrency"]
    model = entry["model_id"]
    ttft = entry["request_throughput"]
    if rate not in grouped:
        grouped[rate] = {}
    grouped[rate][model] = ttft

sorted_rates = sorted(grouped.keys())
all_models = sorted(set(model for group in grouped.values() for model in group))

bar_width = 0.2
x = np.arange(len(sorted_rates))
offsets = np.linspace(-bar_width*(len(all_models)-1)/2, bar_width*(len(all_models)-1)/2, len(all_models))

plt.figure(figsize=(10, 6))

for i, model in enumerate(all_models):
    ttft_values = [grouped[rate].get(model, 0) for rate in sorted_rates]
    bar_positions = x + offsets[i]
    bars = plt.bar(bar_positions, ttft_values, bar_width, label=model.split("/")[-1])
    for pos, val in zip(bar_positions, ttft_values):
        if val > 0:
            plt.text(pos, val + 0.1, f"{val:.2f}", ha='center', va='bottom', fontsize=8)

plt.xlabel("Concorrência Máxima")
plt.ylabel("Request Throughput (req/s)")
plt.title("Request Throughput por Concorrência e Modelo")
plt.xticks(x, sorted_rates)
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.legend(title="Modelo")
plt.tight_layout()
plt.show()